# 05 - Advanced models and hardware

So far we used logistic regression. Real projects compare model families, add their own, and run on
whatever hardware is at hand. This notebook builds that up step by step.

**What you'll learn**
1. how callmefair detects hardware and configures each library (`CALLMEFAIR_DEVICE`);
2. how to compare seven built-in model families with `BiasSearch`;
3. how to plug in your own model with `@register_model`;
4. how to plug a model into the mitigation grid with `register_adapter`;
5. how to audit and mitigate an *intersectional* attribute built with `combine_attributes`;
6. macOS specifics (process start method, OpenMP import order).

**Prerequisites:** notebooks 02 (bias search) and 03 (mitigation grid search).
**Runtime:** about 1 minute on a laptop (no GPU needed). **Data:** stroke prediction.

In [1]:
from fhg_examples import setup; setup()

In [2]:
import os
import subprocess
import sys
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from callmefair import BiasSearch, BMGridSearch, BMInterface, BMType, CType, combine_attributes
from callmefair.util import device
from fhg_examples.datasets import SPECS, load

## 1. Hardware: what is available, and who uses it

`device.detect()` returns `'cuda'`, `'mps'` or `'cpu'` (first match wins: the `CALLMEFAIR_DEVICE`
environment variable, then CUDA, then Apple-Silicon MPS, then CPU). Every library gets its settings from
`callmefair.util.device`, so platform logic lives in one place.

In [3]:
print("detected device:", device.detect(), "| performance cores:", device.performance_cores())
pd.Series({"xgboost": device.xgboost_params(), "catboost": device.catboost_params(),
           "lightgbm": device.lightgbm_params(), "tabnet": device.tabnet_device()},
          name="settings used")

detected device: mps | performance cores: 8


xgboost     {'tree_method': 'hist', 'device': 'cpu'}
catboost                                          {}
lightgbm                                          {}
tabnet                                           cpu
Name: settings used, dtype: object

Only CUDA GPUs speed up the tree libraries; on Apple Silicon XGBoost, CatBoost and LightGBM run on CPU, TabNet
stays on CPU (no measured gain from MPS), and Keras models use a GPU only if TensorFlow sees one.
GPU-backed models are trained one at a time; CPU models are spread over worker processes.

**Forcing a device.** The result is cached, so set `CALLMEFAIR_DEVICE` *before the process starts*, e.g.
`CALLMEFAIR_DEVICE=cpu python script.py`. We can check what a forced value would configure in a fresh interpreter:

In [4]:
probe = "from callmefair.util import device as d; print(d.detect(), d.xgboost_params(), d.catboost_params())"
for forced in ("cuda", "cpu"):
    env = {**os.environ, "CALLMEFAIR_DEVICE": forced}
    out = subprocess.run([sys.executable, "-c", probe], env=env, capture_output=True, text=True).stdout
    print(f"CALLMEFAIR_DEVICE={forced}:", out.strip())

CALLMEFAIR_DEVICE=cuda: cuda {'tree_method': 'hist', 'device': 'cuda'} {'task_type': 'GPU'}
CALLMEFAIR_DEVICE=cpu: cpu {'tree_method': 'hist', 'device': 'cpu'} {}


## 2. The model zoo with `BiasSearch`

`BiasSearch(...).evaluate_average(model_name=...)` trains the named model `iterate` times on different splits and
averages the scores. We use the stroke data and one attribute (`gender`, Male = 1 = reference). `iterate=2`
and `progress=False` keep it quick and quiet; use 10 or more for stable numbers.

In [5]:
stroke = load("stroke")
label = SPECS["stroke"].label
search = BiasSearch(stroke, label, ["gender"], progress=False)

In [6]:
def compare(models):
    rows = []
    for name in models:
        start = time.perf_counter()
        table, _ = search.evaluate_average(iterate=2, model_name=name)
        _, _, fair_score, balanced_acc, *_ = table[1]
        rows.append({"model": name, "balanced_acc": balanced_acc, "fair_score": fair_score,
                     "seconds": time.perf_counter() - start})
    return pd.DataFrame(rows).set_index("model")

zoo = compare(["lr", "xgb", "cat", "lgbm", "mlp", "tabnet", "tabtransformer"])
zoo

,balanced_acc,fair_score,seconds
model,,,
lr,0.773,0.159,2.754
xgb,0.734,0.271,0.174
cat,0.508,0.457,0.189
lgbm,0.647,0.366,0.197
mlp,0.756,0.385,1.267
tabnet,0.701,0.670,3.597
tabtransformer,0.691,0.546,0.894


`fair_score` is normalized to [0, 1] (0 = fair). Differences between models are large here, but with two iterations on
a small data set some of that is noise: run more iterations before ranking models.
Names: `lr` logistic regression, `xgb` XGBoost, `cat` CatBoost, `lgbm` LightGBM, `mlp` Keras MLP, `tabnet` TabNet, `tabtransformer` TabTransformer.

## 3. Your own model with `@register_model`

A *builder* receives `(x, y, sample_weight, n_jobs)`, fits a model and returns a **predictor**: a function from a
feature matrix to the probability of the favorable class. `positive_proba` wraps any scikit-learn classifier for you.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from callmefair.search.model_factory import positive_proba, register_model

@register_model("rf")                       # add uses_gpu=lambda: True for GPU-backed models
def build_rf(x, y, sample_weight, n_jobs):
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=n_jobs, random_state=0)
    return positive_proba(model.fit(x, y, sample_weight=sample_weight))

Worker *processes* start fresh and would not know about a model registered in the notebook, so we ask for **threads**
(`backend="thread"`). Models defined in an importable module can use the default process backend.

In [8]:
search = BiasSearch(stroke, label, ["gender"], backend="thread", progress=False)
compare(["lr", "rf"])

,balanced_acc,fair_score,seconds
model,,,
lr,0.773,0.159,0.068
rf,0.759,0.170,0.491


## 4. Your own model in the mitigation grid with `register_adapter`

`BMGridSearch` talks to models through *adapters*. Anything with `fit` and `predict_proba` already works
(scikit-learn, XGBoost, CatBoost). A model without `predict_proba`, such as `LinearSVC`, is rejected:

In [9]:
from sklearn.svm import LinearSVC
from callmefair.mitigation.model_adapters import register_adapter, resolve_adapter

try:
    resolve_adapter(LinearSVC())
except ValueError as error:
    print("ValueError:", error)

ValueError: Unsupported model type: LinearSVC


An adapter has one method, `fit_predict_proba(model, bmI)`, returning `(val_proba, test_proba)` with shape
`(n_samples, 2)` and the favorable class in column 1. Here we squash the SVM margin with a sigmoid.

In [10]:
class LinearSVCAdapter:
    def fit_predict_proba(self, model, bmI):
        x_train, y_train = bmI.get_train_xy()
        model.fit(x_train, y_train, sample_weight=bmI.get_train_BLD().instance_weights)

        def proba(x):
            p = 1 / (1 + np.exp(-model.decision_function(x)))
            return np.column_stack((1 - p, p))

        return proba(bmI.get_val_xy()[0]), proba(bmI.get_test_xy()[0])

register_adapter(LinearSVC, LinearSVCAdapter())     # matches subclasses too
type(resolve_adapter(LinearSVC())).__name__

'LinearSVCAdapter'

In [11]:
priv, unpriv = SPECS["stroke"].attribute("gender").groups
bm_i = BMInterface(stroke, label=label, protected=["gender"], random_state=0)
grid = BMGridSearch(bm_i, LinearSVC(), [[BMType.preReweighing], [BMType.posEqqOds]], priv, unpriv)
rows = pd.DataFrame(grid.run_single_sensitive(results_dir=None))
rows[["model", "BM", "balanced_acc", "fair_score"]].round(3)

,model,BM,balanced_acc,fair_score
0,LinearSVC,baseline,0.678,0.139
1,LinearSVC,preReweighing,0.678,0.139
2,LinearSVC,posEqqOds,0.666,0.321


## 5. Intersectional attributes

Bias can hide in combinations: a model may treat *older men* differently from both *older people* and *men*.
`combine_attributes` replaces two binary columns by one (`age_gender`) using a set operation (`CType`); with
`intersection` the new attribute is 1 only for people in **both** privileged groups.

In [12]:
age = SPECS["stroke"].attribute("age")
print(f"age = 1 means {age.privileged}; gender = 1 means {SPECS['stroke'].attribute('gender').privileged}")
combined = combine_attributes(stroke, "age", "gender", CType.intersection)
combined["age_gender"].value_counts().rename({1: "privileged (both)", 0: "everyone else"})

age = 1 means 65 or older; gender = 1 means Male


age_gender
everyone else        4692
privileged (both)     417
Name: count, dtype: int64

In [13]:
priv, unpriv = [{"age_gender": 1}], [{"age_gender": 0}]
bm_i = BMInterface(combined, label=label, protected=["age_gender"], random_state=0)
bm_list = [[BMType.preReweighing], [BMType.posEqqOds], [BMType.posROC]]
grid = BMGridSearch(bm_i, LogisticRegression(max_iter=1000), bm_list, priv, unpriv)
rows = pd.DataFrame(grid.run_single_sensitive(results_dir=None))
rows[["BM", "balanced_acc", "spd", "fair_score"]].round(3)

,BM,balanced_acc,spd,fair_score
0,baseline,0.687,-0.748,1.000
1,preReweighing,0.626,0.001,0.713
2,posEqqOds,0.502,-0.019,0.107
3,posROC,0.628,0.034,0.632


The grid works unchanged on the new attribute: the same code, the same metrics, only `protected` and the groups differ.
Notebook 04 turns such a table into a trade-off map.

## 6. macOS notes

- **Spawn start method.** macOS starts worker processes by *spawn* (a fresh interpreter that re-imports your script),
  so scripts must put their entry point under a guard. Notebooks need nothing special.

  ```python
  def main():
      ...  # BiasSearch / BMGridSearch calls

  if __name__ == "__main__":
      main()
  ```
- **LightGBM and PyTorch.** If PyTorch loads its OpenMP runtime before LightGBM, multi-threaded LightGBM can crash
  later in the same process. Use `device.import_torch()` instead of `import torch` (it loads LightGBM first on macOS).

In [14]:
torch = device.import_torch()
torch.__version__

'2.2.2'

## Summary

- `device.detect()` and the per-library helpers decide where models run; `CALLMEFAIR_DEVICE` overrides it before start-up.
- `BiasSearch(..., model_name=...)` compares seven built-in families; add more with `@register_model` and `positive_proba`.
- `register_adapter` teaches `BMGridSearch` about models without `predict_proba`.
- `combine_attributes` + `BMGridSearch` audit intersectional groups with no other changes.

**Next:** back to [04_tradeoff_map.ipynb](04_tradeoff_map.ipynb) to place any of these results on the trade-off map.